[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MarcusBae/EYE-D/blob/feat/phase1/edge/notebooks/debug_yolo.ipynb)

# Debug: YOLO 추론 문제 진단

data_960 영상에서 사람이 있는데 YOLO가 감지 못하는 문제 디버깅

## 1. 라이브러리 로드

In [ ]:
import subprocess, sys

for pkg in ['ultralytics', 'boxmot']:
    try:
        __import__(pkg)
    except ImportError:
        print(f'설치 중: {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print('패키지 설치 완료')

## 1. 라이브러리 로드

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
import torch

DEVICE = '0' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA device: {torch.cuda.get_device_name(0)}')

## 2. 모델 로드

In [ ]:
detector = YOLO('yolov8n.pt')
print(f'모델 로드 완료: {detector.model}')
print(f'모델 디바이스: {next(detector.model.parameters()).device}')

## 3. 한 프레임 로드 및 테스트

In [ ]:
# ── 영상 경로 (Colab 또는 로컬) ──────────────────────────────────────────
import os

if os.environ.get('COLAB_RELEASE_TAG'):
    VIDEO_PATH = '/content/drive/MyDrive/projects/EYE-D/EYE-D/data_960/16000000.avi'
else:
    VIDEO_PATH = 'data_960/16000000.avi'

print(f'영상 경로: {VIDEO_PATH}')
print(f'파일 존재: {os.path.exists(VIDEO_PATH)}')

# ── 10번째 프레임 로드 ──────────────────────────────────────────────────
cap = cv2.VideoCapture(VIDEO_PATH)
cap.set(cv2.CAP_PROP_POS_FRAMES, 10)
ret, frame = cap.read()
cap.release()

if ret:
    print(f'프레임 로드 성공')
    print(f'  형태: {frame.shape}')
    print(f'  dtype: {frame.dtype}')
    print(f'  범위: [{frame.min()}, {frame.max()}]')
else:
    print('프레임 로드 실패!')

## 4. YOLO 단일 프레임 추론

In [ ]:
# ── 단일 프레임 추론 ──────────────────────────────────────────────────────
print('단일 프레임으로 YOLO 추론...')
result_single = detector.predict(frame, conf=0.40, classes=[0], verbose=True)
print(f'\n결과:')
print(f'  감지 수: {len(result_single[0].boxes)}')
if len(result_single[0].boxes) > 0:
    print(f'  신뢰도: {result_single[0].boxes.conf.cpu().numpy()}')
    print(f'  bbox: {result_single[0].boxes.xyxy.cpu().numpy()}')

## 5. YOLO 배치 추론 (collect_tracks 방식)

In [ ]:
# ── 4개 프레임으로 배치 추론 ────────────────────────────────────────────
cap = cv2.VideoCapture(VIDEO_PATH)
frames = []
for i in range(4):
    cap.set(cv2.CAP_PROP_POS_FRAMES, 10 + i * 5)
    ret, frame = cap.read()
    if ret:
        frames.append(frame)
cap.release()

print(f'로드된 프레임: {len(frames)}개')
if frames:
    print(f'  형태: {frames[0].shape}')
    print(f'  타입: {type(frames[0])}')

# ── 배치 추론 ──────────────────────────────────────────────────────────
print('\n배치 추론...')
batch_results = detector.predict(frames, conf=0.40, classes=[0], verbose=True)
print(f'\n배치 결과:')
for i, res in enumerate(batch_results):
    n = len(res.boxes)
    print(f'  프레임 {i}: {n}개 감지')
    if n > 0:
        print(f'    신뢰도: {res.boxes.conf.cpu().numpy()}')

## 6. CONF_THRESH 변경해서 테스트

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)
cap.set(cv2.CAP_PROP_POS_FRAMES, 10)
ret, frame = cap.read()
cap.release()

print('임계값별 감지 수:')
for conf_thresh in [0.10, 0.20, 0.30, 0.40, 0.50]:
    result = detector.predict(frame, conf=conf_thresh, classes=[0], verbose=False)
    n = len(result[0].boxes)
    print(f'  CONF={conf_thresh}: {n}개 감지')
    if n > 0:
        print(f'    신뢰도 범위: [{result[0].boxes.conf.min():.3f}, {result[0].boxes.conf.max():.3f}]')